<a href="https://colab.research.google.com/github/Shineii86/RepoGardener/blob/main/notebooks/RepoGardener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/RepoGardener/main/images/RepoGardener.png" width="300px" alt="Repo Gardener">
  <h1>🌱 Repo Gardener (Multi‑Ecosystem)</h1>
  <p><b>Automatically scan, audit, and update dependencies across Python, Node.js, Java, and Go projects.</b></p>
</div>

---

> ℹ️ **ABOUT THIS TOOL**
> - This script scans all your GitHub repositories for outdated dependencies across multiple package managers and can optionally open pull requests to update them.
> - **Supported ecosystems**: Python (`requirements.txt`), Node.js (`package.json`), Java (`pom.xml`), Go (`go.mod`).
> - You need a **Personal Access Token (classic)** with `repo` and `workflow` scopes.
> - This is a powerful automation tool. Use it responsibly and review changes before merging.

---

In [ ]:
#@title 📦 1. Install Dependencies & Setup
!pip install -q PyGithub requests packaging xmltodict

import os
import re
import time
import base64
import json
import requests
import xmltodict
from datetime import datetime
from github import Github, GithubException
from packaging import version
from abc import ABC, abstractmethod

print("✅ All dependencies installed and imported.")

In [ ]:
#@title ⚙️ 2. Configuration & Run

# =============================================================================
#  🔧 Configuration
# =============================================================================

# Your GitHub username
USERNAME = "Shineii86"  #@param {type:"string"}

# Your Personal Access Token (classic) with 'repo' and 'workflow' scopes
TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Mode: "scan" (report only) or "update" (scan and create PRs)
MODE = "scan"  #@param ["scan", "update"] {type:"string"}

# Repository filter (leave blank to scan all, or use comma-separated list)
REPO_FILTER = ""  #@param {type:"string"}

# Ecosystems to scan (comma-separated: python,node,java,go)
ECOSYSTEMS = "python,node,java,go"  #@param {type:"string"}

# Delay between API calls (seconds) to avoid rate limits
ACTION_DELAY = 1  #@param {type:"number"}

# =============================================================================
#  🌱 Package Manager Abstraction
# =============================================================================

class PackageManager(ABC):
    """Abstract base class for package managers."""

    @abstractmethod
    def name(self) -> str:
        pass

    @abstractmethod
    def filename(self) -> str:
        pass

    @abstractmethod
    def parse_dependencies(self, content: str) -> dict:
        """Return dict of package -> current_version."""
        pass

    @abstractmethod
    def get_latest_version(self, package_name: str) -> str:
        """Query registry for latest version."""
        pass

    def update_file(self, content: str, outdated: dict) -> str:
        """Update the file content with new versions. Override if needed."""
        return content

    def check_outdated(self, packages: dict) -> dict:
        """Return dict of outdated packages with current and latest versions."""
        outdated = {}
        for pkg, current_ver in packages.items():
            if current_ver == "latest":
                continue
            latest = self.get_latest_version(pkg)
            if latest:
                try:
                    if version.parse(latest) > version.parse(current_ver):
                        outdated[pkg] = {"current": current_ver, "latest": latest}
                except:
                    pass
        return outdated

# -----------------------------------------------------------------------------
#  🐍 Python (requirements.txt)
# -----------------------------------------------------------------------------

class PythonManager(PackageManager):
    def name(self):
        return "Python"

    def filename(self):
        return "requirements.txt"

    def parse_dependencies(self, content):
        packages = {}
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            match = re.match(r'^([a-zA-Z0-9_\-]+)([=~<>!]+)([a-zA-Z0-9_.\-]+)', line)
            if match:
                pkg, op, ver = match.groups()
                packages[pkg.lower()] = ver
            else:
                pkg_match = re.match(r'^([a-zA-Z0-9_\-]+)', line)
                if pkg_match:
                    packages[pkg_match.group(1).lower()] = "latest"
        return packages

    def get_latest_version(self, package_name):
        try:
            url = f"https://pypi.org/pypi/{package_name}/json"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return response.json()["info"]["version"]
        except:
            pass
        return None

    def update_file(self, content, outdated):
        lines = content.split('\n')
        new_lines = []
        for line in lines:
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                new_lines.append(line)
                continue
            match = re.match(r'^([a-zA-Z0-9_\-]+)([=~<>!]+)([a-zA-Z0-9_.\-]+)', stripped)
            if match:
                pkg, op, ver = match.groups()
                pkg_lower = pkg.lower()
                if pkg_lower in outdated:
                    new_ver = outdated[pkg_lower]["latest"]
                    new_line = f"{pkg}{op}{new_ver}"
                    new_lines.append(new_line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
        return '\n'.join(new_lines)

# -----------------------------------------------------------------------------
#  🟢 Node.js (package.json)
# -----------------------------------------------------------------------------

class NodeManager(PackageManager):
    def name(self):
        return "Node.js"

    def filename(self):
        return "package.json"

    def parse_dependencies(self, content):
        packages = {}
        try:
            data = json.loads(content)
            for dep_type in ['dependencies', 'devDependencies']:
                if dep_type in data:
                    for pkg, ver in data[dep_type].items():
                        clean_ver = re.sub(r'[\^~><=]', '', ver.strip())
                        if clean_ver and clean_ver != '*':
                            packages[pkg] = clean_ver
        except:
            pass
        return packages

    def get_latest_version(self, package_name):
        try:
            url = f"https://registry.npmjs.org/{package_name}/latest"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return response.json()["version"]
        except:
            pass
        return None

    def update_file(self, content, outdated):
        try:
            data = json.loads(content)
            for dep_type in ['dependencies', 'devDependencies']:
                if dep_type in data:
                    for pkg in data[dep_type]:
                        if pkg in outdated:
                            data[dep_type][pkg] = f"^{outdated[pkg]['latest']}"
            return json.dumps(data, indent=2)
        except:
            return content

# -----------------------------------------------------------------------------
#  ☕ Java (pom.xml)
# -----------------------------------------------------------------------------

class JavaManager(PackageManager):
    def name(self):
        return "Java/Maven"

    def filename(self):
        return "pom.xml"

    def parse_dependencies(self, content):
        packages = {}
        try:
            data = xmltodict.parse(content)
            deps = data.get('project', {}).get('dependencies', {}).get('dependency', [])
            if isinstance(deps, dict):
                deps = [deps]
            for dep in deps:
                if 'version' in dep and 'artifactId' in dep:
                    pkg = f"{dep.get('groupId', '')}:{dep['artifactId']}"
                    ver = dep['version']
                    if not ver.startswith('$'):  # skip property placeholders
                        packages[pkg] = ver
        except:
            pass
        return packages

    def get_latest_version(self, package_name):
        try:
            group, artifact = package_name.split(':')
            url = f"https://search.maven.org/solrsearch/select?q=g:{group}+AND+a:{artifact}&rows=1&wt=json"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                docs = response.json().get('response', {}).get('docs', [])
                if docs:
                    return docs[0].get('latestVersion', docs[0].get('v'))
        except:
            pass
        return None

    def update_file(self, content, outdated):
        # Simple regex-based update for pom.xml
        for pkg, info in outdated.items():
            group, artifact = pkg.split(':')
            pattern = rf'(<groupId>{group}</groupId>\s*<artifactId>{artifact}</artifactId>\s*<version>)[^<]+(</version>)'
            replacement = rf'\g<1>{info["latest"]}\g<2>'
            content = re.sub(pattern, replacement, content, flags=re.DOTALL)
        return content

# -----------------------------------------------------------------------------
#  🐹 Go (go.mod)
# -----------------------------------------------------------------------------

class GoManager(PackageManager):
    def name(self):
        return "Go"

    def filename(self):
        return "go.mod"

    def parse_dependencies(self, content):
        packages = {}
        lines = content.split('\n')
        in_require = False
        for line in lines:
            line = line.strip()
            if line.startswith('require ('):
                in_require = True
                continue
            if in_require:
                if line == ')':
                    in_require = False
                    continue
                parts = line.split()
                if len(parts) >= 2:
                    pkg = parts[0]
                    ver = parts[1]
                    if not ver.startswith('//'):
                        packages[pkg] = ver
            elif line.startswith('require '):
                parts = line.split()
                if len(parts) >= 3:
                    pkg = parts[1]
                    ver = parts[2]
                    packages[pkg] = ver
        return packages

    def get_latest_version(self, package_name):
        try:
            url = f"https://proxy.golang.org/{package_name}/@latest"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                data = response.json()
                return data.get("Version")
        except:
            pass
        return None

    def update_file(self, content, outdated):
        lines = content.split('\n')
        new_lines = []
        in_require = False
        for line in lines:
            stripped = line.strip()
            if stripped.startswith('require ('):
                in_require = True
                new_lines.append(line)
                continue
            if in_require:
                if stripped == ')':
                    in_require = False
                    new_lines.append(line)
                    continue
                parts = stripped.split()
                if len(parts) >= 2:
                    pkg = parts[0]
                    if pkg in outdated:
                        indent = line[:len(line) - len(line.lstrip())]
                        new_line = f"{indent}{pkg} {outdated[pkg]['latest']}"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)
                else:
                    new_lines.append(line)
            elif stripped.startswith('require '):
                parts = stripped.split()
                if len(parts) >= 3:
                    pkg = parts[1]
                    if pkg in outdated:
                        new_line = f"require {pkg} {outdated[pkg]['latest']}"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
        return '\n'.join(new_lines)

# =============================================================================
#  🛠️ Utility Functions
# =============================================================================

def get_managers(ecosystems_str):
    """Return list of package manager instances based on selection."""
    managers = []
    selected = [e.strip().lower() for e in ecosystems_str.split(',')]
    if 'python' in selected:
        managers.append(PythonManager())
    if 'node' in selected:
        managers.append(NodeManager())
    if 'java' in selected:
        managers.append(JavaManager())
    if 'go' in selected:
        managers.append(GoManager())
    return managers

def get_file_content(repo, path):
    """Get the content of a file from a repository."""
    try:
        content = repo.get_contents(path)
        return base64.b64decode(content.content).decode('utf-8')
    except GithubException:
        return None

def create_update_branch(repo, branch_name, base_sha):
    """Create a new branch for updates."""
    try:
        repo.create_git_ref(f"refs/heads/{branch_name}", base_sha)
        return True
    except GithubException:
        try:
            repo.get_git_ref(f"heads/{branch_name}").delete()
            repo.create_git_ref(f"refs/heads/{branch_name}", base_sha)
            return True
        except:
            return False

def update_file_on_branch(repo, path, new_content, branch_name, commit_msg):
    """Update a file on a specific branch."""
    try:
        file = repo.get_contents(path, ref=branch_name)
        repo.update_file(path, commit_msg, new_content, file.sha, branch=branch_name)
        return True
    except GithubException:
        return False

def create_pull_request(repo, branch_name, title, body):
    """Create a pull request."""
    try:
        pr = repo.create_pull(
            title=title,
            body=body,
            head=branch_name,
            base=repo.default_branch
        )
        return pr
    except GithubException as e:
        print(f"  ⚠️ Failed to create PR: {e}")
        return None

# =============================================================================
#  🚀 Main Execution
# =============================================================================

print(f"\n🌱 Repo Gardener (Multi‑Ecosystem) for user '{USERNAME}'")
print(f"Mode: {MODE.upper()}")
print(f"Ecosystems: {ECOSYSTEMS}")
print("="*50)

# Initialize GitHub client
g = Github(TOKEN)
user = g.get_user(USERNAME)

# Get package managers
managers = get_managers(ECOSYSTEMS)
if not managers:
    print("❌ No valid ecosystems selected. Exiting.")
    exit()

# Get repositories to scan
if REPO_FILTER.strip():
    filter_list = [r.strip() for r in REPO_FILTER.split(',')]
    repos = [repo for repo in user.get_repos() if repo.name in filter_list]
else:
    repos = list(user.get_repos())

print(f"📂 Found {len(repos)} repositories to scan.\n")

scan_results = []

for repo in repos:
    print(f"🔍 Scanning {repo.full_name}...")
    
    if repo.archived:
        print(f"  ⏭️ Skipping archived repository.")
        continue
    
    repo_outdated = {}
    
    for manager in managers:
        file_path = manager.filename()
        content = get_file_content(repo, file_path)
        if not content:
            continue
        
        packages = manager.parse_dependencies(content)
        outdated = manager.check_outdated(packages)
        
        if outdated:
            print(f"  📦 [{manager.name()}] Found {len(outdated)} outdated package(s):")
            for pkg, versions in outdated.items():
                print(f"     - {pkg}: {versions['current']} → {versions['latest']}")
            repo_outdated[manager] = (file_path, outdated, content)
        else:
            print(f"  ✅ [{manager.name()}] All dependencies up-to-date.")
    
    if repo_outdated:
        scan_results.append({
            "repo": repo,
            "outdated": repo_outdated
        })
        
        if MODE == "update":
            # Create a single branch and PR for all updates in this repo
            main_branch = repo.get_branch(repo.default_branch)
            base_sha = main_branch.commit.sha
            branch_name = f"repo-gardener/update-deps-{datetime.now().strftime('%Y%m%d%H%M%S')}"
            
            if not create_update_branch(repo, branch_name, base_sha):
                print(f"  ❌ Failed to create branch {branch_name}. Skipping updates.")
                continue
            
            pr_body = "This PR updates outdated dependencies:\n\n"
            success = True
            
            for manager, (file_path, outdated, original_content) in repo_outdated.items():
                new_content = manager.update_file(original_content, outdated)
                commit_msg = f"🌱 Update {manager.name()} dependencies"
                if update_file_on_branch(repo, file_path, new_content, branch_name, commit_msg):
                    pr_body += f"### {manager.name()}\n"
                    for pkg, versions in outdated.items():
                        pr_body += f"- `{pkg}`: {versions['current']} → {versions['latest']}\n"
                    pr_body += "\n"
                else:
                    print(f"  ❌ Failed to update {file_path}")
                    success = False
            
            if success:
                pr_title = "🌱 Repo Gardener: Update dependencies"
                pr_body += "\n*This PR was automatically created by [Repo Gardener](https://github.com/Shineii86/RepoGardener).*"
                pr = create_pull_request(repo, branch_name, pr_title, pr_body)
                if pr:
                    print(f"  🎉 Created PR: {pr.html_url}")
            else:
                print(f"  ❌ Some updates failed; no PR created.")
    else:
        print(f"  ✨ No outdated dependencies found in any ecosystem.")
    
    time.sleep(ACTION_DELAY)

print("\n" + "="*50)

if not scan_results:
    print("✨ No outdated dependencies found across all repositories!")
else:
    print(f"📊 Summary: Found outdated dependencies in {len(scan_results)} repository(ies).")
    for result in scan_results:
        repo = result["repo"]
        print(f"\n📁 {repo.full_name}:")
        for manager, (_, outdated, _) in result["outdated"].items():
            print(f"  [{manager.name()}]")
            for pkg, versions in outdated.items():
                print(f"    - {pkg}: {versions['current']} → {versions['latest']}")

print("\n---")
print("🌱 Repo Gardener By [Shinei Nouzen](https://github.com/Shineii86/RepoGardener)")


---

### 📚 How It Works

1. **Scan**: The script iterates through all your GitHub repositories (or a filtered list).
2. **Detect**: For each repository, it looks for dependency files (`requirements.txt`, `package.json`, `pom.xml`, `go.mod`).
3. **Parse**: It extracts package names and current versions using the appropriate parser.
4. **Compare**: It queries the relevant registry (PyPI, npm, Maven Central, Go proxy) for the latest version.
5. **Update**: In `update` mode, it creates a new branch, updates all outdated files, and opens a single pull request per repository.

### 🛠️ Adding More Ecosystems

To add support for a new package manager:
1. Create a new class inheriting from `PackageManager`.
2. Implement `name()`, `filename()`, `parse_dependencies()`, `get_latest_version()`, and optionally `update_file()`.
3. Add it to the `get_managers()` function.

### 📅 Scheduling

To run this as a weekly health check:
1. In Colab, go to **Tools → Command Palette** (Ctrl+Shift+P).
2. Search for "Run on schedule".
3. Set a schedule (e.g., every Monday at 9 AM).
4. Save the notebook to your Google Drive.

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for developers who want to keep their code healthy*


***Found this useful? Give it a Star! ⭐ [Shineii86/RepoGardener](https://github.com/Shineii86/RepoGardener)***
</div>